# Chapter 6 — Project Walkthrough

**Time:** ~1 hour  
**Goal:** Read and understand every file in the Smart Energy Optimization System. Map what you learned in Part 1 to real code.

---

## Before You Start

Make sure you have the project on your machine. Open the project folder in VS Code alongside this notebook.

The project structure:

```
smart-energy-optimization-system/
└── src/
    ├── config.py      ← Constants (Chapter 2)
    ├── sensor.py      ← SensorSimulator class (Chapter 5)
    ├── engine.py      ← DecisionEngine class (Chapter 3 + 5)
    ├── calculator.py  ← EnergyCalculator class (Chapter 4 + 5)
    ├── logger.py      ← DataLogger class (Chapter 5)
    ├── main.py        ← Wires everything together (Chapter 4)
    └── dashboard.py   ← Streamlit UI (Chapter 7)
```

---

## 6.1 The Pipeline

Before reading any file, understand how data flows through the system:

```
SensorSimulator
    ↓  get_data()  →  {temperature, humidity, occupancy, hour}
DecisionEngine
    ↓  evaluate()  →  ac_state (True / False)
EnergyCalculator
    ↓  compute()   →  cycle_energy (float)
DataLogger
    ↓  store()     →  record saved in memory
Dashboard / Terminal
       displays the record
```

Each class does one job and passes its output to the next. Nothing does more than its job.

## 6.2 config.py — All Constants in One Place

Open `src/config.py`. Every value here is a variable (Chapter 2) that configures the entire system.

```python
TEMP_MIN       = 18.0
TEMP_MAX       = 40.0
TEMP_INIT      = 28.0

DEFAULT_THRESHOLD = 26.0
DEFAULT_BUFFER    = 1.5

RATE_ON  = 2.0
RATE_OFF = 0.1
```

**Why a separate file for constants?**  
Every other file imports from here. If you want to change the temperature threshold, you change it in one place — `config.py` — and the effect propagates everywhere automatically. You never have to search through 7 files to find where `26.0` is hardcoded.

**What `from config import ...` means:**  
Each file starts with lines like:
```python
from config import DEFAULT_THRESHOLD, DEFAULT_BUFFER
```
This makes those constants available inside that file. You will see this pattern in every source file.

## 6.3 sensor.py — SensorSimulator

Open `src/sensor.py`.

```python
class SensorSimulator:
    def __init__(self):
        self.last_temp = TEMP_INIT      # starts at 28.0°C

    def _clamp(self, value, min_val, max_val):
        return max(min_val, min(max_val, value))

    def _next_temperature(self, ac_state):
        if ac_state:
            drift = random.uniform(DRIFT_AC_ON_MIN, DRIFT_AC_ON_MAX)   # cooling
        else:
            drift = random.uniform(DRIFT_AC_OFF_MIN, DRIFT_AC_OFF_MAX) # heating
        new_temp = self._clamp(self.last_temp + drift, TEMP_MIN, TEMP_MAX)
        self.last_temp = new_temp
        return round(new_temp, 1)

    def get_data(self, ac_state=False):
        return {
            "temperature": self._next_temperature(ac_state),
            "humidity":    round(random.uniform(HUMIDITY_MIN, HUMIDITY_MAX), 1),
            "occupancy":   random.choices([1, 0], weights=[70, 30])[0],
            "hour":        datetime.now().hour,
        }
```

**Key observations:**

1. `__init__` stores `last_temp` — this persists across calls so temperature drifts realistically, not randomly jumping
2. `_clamp` is a helper — the underscore prefix `_` is a Python convention meaning "this is internal, not meant to be called from outside"
3. `_next_temperature` implements a feedback loop: if AC was ON last cycle, room tends to cool; if OFF, it tends to heat
4. `get_data` returns a **dictionary** — a data structure that stores key-value pairs
5. `occupancy` uses `weights=[70, 30]` — room is occupied 70% of the time, empty 30%

### What is a Dictionary?

A dictionary stores data as `key: value` pairs, like a real dictionary stores `word: definition`.

In [ ]:
# Creating a dictionary
sensor_data = {
    "temperature": 29.5,
    "humidity":    68.2,
    "occupancy":   1,
    "hour":        14,
}

# Accessing values by key
print(sensor_data["temperature"])   # 29.5
print(sensor_data["occupancy"])     # 1

## 6.4 engine.py — DecisionEngine

Open `src/engine.py`.

This class applies 5 rules in priority order to decide whether AC should be ON or OFF:

```python
class DecisionEngine:
    def __init__(self):
        self.ac_state    = False
        self.last_reason = "Simulation starting..."

    def evaluate(self, data, threshold=26.0, buffer=1.5, ...):
        temp      = data["temperature"]
        occupancy = data["occupancy"]
        hour      = data["hour"]
        upper     = threshold + buffer   # 27.5
        lower     = threshold - buffer   # 24.5

        if occupancy == 0:              # Rule 1: empty room → OFF
            self.ac_state = False
        elif not (op_start <= hour < op_end):  # Rule 2: wrong time → OFF
            self.ac_state = False
        elif temp > upper:              # Rule 3: too hot → ON
            self.ac_state = True
        elif temp < lower:              # Rule 4: cool enough → OFF
            self.ac_state = False
        # Rule 5: in the band → hold current state (no change)

        return self.ac_state
```

**Key concept — Hysteresis:**  
Instead of a single threshold (AC ON above 26°C, OFF below 26°C), the engine uses a band:
- AC turns ON when temperature exceeds **27.5°C** (upper = threshold + buffer)
- AC turns OFF when temperature falls below **24.5°C** (lower = threshold − buffer)
- Between 24.5°C and 27.5°C → hold current state

This prevents rapid on/off switching when temperature hovers near the threshold. It is the same principle used in real thermostats.

**`last_reason`** stores a human-readable explanation of why the last decision was made. The dashboard displays this.

## 6.5 calculator.py — EnergyCalculator

Open `src/calculator.py`.

```python
class EnergyCalculator:
    def __init__(self):
        self.cumulative = 0.0
        self.rate_on    = RATE_ON    # 2.0 units per cycle
        self.rate_off   = RATE_OFF   # 0.1 units per cycle (standby power)

    def compute(self, ac_state):
        cycle_energy     = self.rate_on if ac_state else self.rate_off
        self.cumulative += cycle_energy
        return round(cycle_energy, 2)

    def get_cumulative(self):
        return round(self.cumulative, 2)

    def reset(self):
        self.cumulative = 0.0
```

**Key observations:**
1. Even when AC is OFF, `RATE_OFF = 0.1` is consumed — this models standby/idle power draw
2. `self.cumulative` accumulates across the entire simulation — this is stored state (Chapter 5.5)
3. The expression `self.rate_on if ac_state else self.rate_off` is a **ternary expression** — a compact if/else in one line

In [ ]:
# Ternary expression — same as if/else in one line
ac_state = True

# This:
energy = 2.0 if ac_state else 0.1

# Is exactly the same as:
if ac_state:
    energy = 2.0
else:
    energy = 0.1

print(energy)

## 6.6 logger.py — DataLogger

Open `src/logger.py`.

```python
class DataLogger:
    def __init__(self):
        self.records = []   # empty list

    def store(self, record):
        self.records.append(record)   # add one record to the list

    def get_latest(self, n=50):
        return self.records[-n:]  # return last n records

    def count(self):
        return len(self.records)
```

This is the simplest class in the project. A list (`[]`) that grows with every cycle. The dashboard reads from it using `get_latest()`.

## 6.7 main.py — How Everything Connects

Open `src/main.py`. This is the entry point for the terminal runner. Read it alongside this explanation.

```python
def main():
    # Create one instance of each class
    simulator  = SensorSimulator()
    engine     = DecisionEngine()
    calculator = EnergyCalculator()
    logger     = DataLogger()

    while True:                              # run forever
        data = simulator.get_data(engine.ac_state)     # Step 1: sense
        ac_state = engine.evaluate(data, ...)          # Step 2: decide
        cycle_energy = calculator.compute(ac_state)    # Step 3: calculate

        record = { ... }                     # Step 4: build a record dict
        logger.store(record)                 # Step 5: log it

        print(...)                           # Step 6: display
        time.sleep(REFRESH_INTERVAL)         # Step 7: wait 2 seconds
```

**Note the feedback loop:**  
The sensor receives `engine.ac_state` — the current AC state — so it knows to drift temperature up or down in the next cycle. This creates a realistic simulation where AC actually affects temperature.

## 6.8 Run the Terminal Simulation

Open your terminal. Navigate to the project folder:

```bash
cd path/to/smart-energy-optimization-system
pip install -r requirements.txt
python3 src/main.py
```

You should see output like:

```
Time                   Temp   Hum  Occ   AC   Cycle    Total
---------------------------------------------------------------------------
2024-01-15 14:30:01   29.5°C  68.2%  Yes    ON     2.0      2.0
2024-01-15 14:30:03   27.8°C  71.0%  Yes    ON     2.0      4.0
2024-01-15 14:30:05   25.1°C  65.4%  Yes   OFF     0.1      4.1
```

Trace what happens each line:
1. Sensor generates temperature/humidity/occupancy
2. Engine evaluates and sets AC state
3. Calculator adds energy for this cycle
4. Logger stores the record
5. One row prints
6. Program waits 2 seconds

Press `Ctrl+C` to stop.

---
## Exercises

**Exercise 1:** In `engine.py`, what are the exact temperature values where AC turns ON and OFF with the default settings (`threshold=26.0`, `buffer=1.5`)? Write the numbers, do not guess.

*Your answer here:*  
AC turns ON when temperature > ___°C  
AC turns OFF when temperature < ___°C

**Exercise 2:** Open `config.py` and change `DEFAULT_THRESHOLD` to `28.0`. Run `main.py` again. Describe what changed in the output and explain why.

*Your answer here:*

**Exercise 3:** In `main.py`, find the line that passes `engine.ac_state` to `simulator.get_data()`. Explain in your own words why this is necessary — what would happen if you always passed `False` instead?

*Your answer here:*

---
**Chapter 6 complete.** Move on to Chapter 7 — The Dashboard.